# Section 1: Configuration & Authentication

First, let's set up our configuration variables and verify authentication.

In [4]:
# Configuration variables

PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           


print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357


In [5]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


# Section 2: Create BigQuery Dataset and Tables


## Copy Census Bureau ACS Dataset

This section copies all tables from the public BigQuery dataset `bigquery-public-data.census_bureau_acs` into your project.

**What's included:**
- ~280 tables containing US Census Bureau American Community Survey data
- Tables include blockgroup, cbsa (Core Based Statistical Area), and census tract data
- Data spans multiple years (2007-2018) and survey periods (1-year, 3-year, 5-year)

**Process:**
1. Install required libraries
2. Create destination dataset in your project
3. List all tables from source dataset
4. Copy tables individually (more reliable than Data Transfer Service)
5. Validate completion

**Estimated time:** 10-30 minutes depending on number of tables

**Note:** We use direct table copying rather than Data Transfer Service for simpler setup without requiring service account configuration.

In [12]:
# Install required libraries for BigQuery Data Transfer
import sys
import subprocess

# Install the packages
packages = ["google-cloud-bigquery-datatransfer", "google-cloud-bigquery"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed packages are available.")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed

⚠️  IMPORTANT: After running this cell, please restart the kernel:
   - Click 'Kernel' → 'Restart' in the menu
   - Or use the restart button in the toolbar
   - Then continue running from the next cell

   This ensures the newly installed packages are available.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
# Alternative: Reload packages without kernel restart (try this first)
import sys
import importlib

# Force reload the package path
if 'google.cloud.bigquery_datatransfer' in sys.modules:
    del sys.modules['google.cloud.bigquery_datatransfer']
if 'google.cloud.bigquery' in sys.modules:
    del sys.modules['google.cloud.bigquery']

# Now import
try:
    from google.cloud import bigquery
    from google.cloud import bigquery_datatransfer
    
    # Initialize BigQuery client
    bq_client = bigquery.Client(project=PROJECT_ID)
    
    # Dataset configuration
    DATASET_ID = "census_bureau_acs"
    DATASET_LOCATION = "US"  # Must match source dataset location
    
    # Create dataset
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = DATASET_LOCATION
    
    dataset = bq_client.create_dataset(dataset, exists_ok=True)
    print(f"✅ Dataset created/verified: {dataset_id}")
    print(f"   Location: {DATASET_LOCATION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Then run cells 1-3 again (config and auth)")
    print("   - Skip cell 6 (already installed)")
    print("   - Continue from this cell")
    raise

✅ Dataset created/verified: johnswain-1200-20260303091357.census_bureau_acs
   Location: US


In [14]:
# List all tables in source dataset to copy
# Using direct table copying instead of Data Transfer Service (simpler setup)

SOURCE_PROJECT_ID = "bigquery-public-data"
SOURCE_DATASET_ID = "census_bureau_acs"
source_dataset_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}"

print(f"📋 Preparing to copy tables:")
print(f"   Source: {source_dataset_ref}")
print(f"   Destination: {dataset_id}")
print()
print("🔍 Listing tables in source dataset...")

# List all tables in the source dataset
source_tables = list(bq_client.list_tables(source_dataset_ref))
total_tables = len(source_tables)

print(f"   Found {total_tables} tables to copy")
print()

# Show first 10 as preview
print("📋 First 10 tables:")
for i, table in enumerate(source_tables[:10]):
    print(f"   {i+1}. {table.table_id}")
if total_tables > 10:
    print(f"   ... and {total_tables - 10} more")

print()
print(f"✅ Ready to copy {total_tables} tables")

📋 Transfer Configuration:
   Source: bigquery-public-data.census_bureau_acs
   Destination: johnswain-1200-20260303091357.census_bureau_acs
   Tables to copy: ~280 tables
❌ Error creating transfer config: 400 Failed to find a valid credential. The field 'version_info' or 'service_account_name' must be specified.


InvalidArgument: 400 Failed to find a valid credential. The field 'version_info' or 'service_account_name' must be specified.

In [ ]:
# Copy all tables from source to destination
import time

print("🚀 Starting table copy operations...")
print(f"   Copying {total_tables} tables")
print()

# Track progress
copied_tables = []
failed_tables = []

# Copy tables in batches
for i, source_table in enumerate(source_tables):
    table_name = source_table.table_id
    source_table_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}.{table_name}"
    dest_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    
    try:
        # Create copy job
        job_config = bigquery.CopyJobConfig()
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        
        copy_job = bq_client.copy_table(
            source_table_ref,
            dest_table_ref,
            job_config=job_config
        )
        
        # Wait for job to complete (with timeout)
        copy_job.result(timeout=300)  # 5 minute timeout per table
        
        copied_tables.append(table_name)
        
        # Print progress every 10 tables
        if (i + 1) % 10 == 0:
            print(f"   ✅ Copied {i + 1}/{total_tables} tables...")
        
    except Exception as e:
        print(f"   ❌ Failed to copy {table_name}: {str(e)[:100]}")
        failed_tables.append((table_name, str(e)))
        continue

print()
print("=" * 60)
print(f"✅ Copy operation completed!")
print(f"   Successfully copied: {len(copied_tables)} tables")
if failed_tables:
    print(f"   Failed: {len(failed_tables)} tables")
print("=" * 60)

In [ ]:
# Show any failed tables
if failed_tables:
    print()
    print("⚠️  Failed Tables:")
    for table_name, error in failed_tables[:10]:  # Show first 10
        print(f"   - {table_name}: {error[:80]}...")
    if len(failed_tables) > 10:
        print(f"   ... and {len(failed_tables) - 10} more failures")
else:
    print()
    print("✅ All tables copied successfully!")

In [ ]:
# Validate dataset copy
print("🔍 Validating copied dataset...")
print()

# List tables in destination dataset
dest_tables = list(bq_client.list_tables(dataset_id))
table_count = len(dest_tables)

print(f"✅ Destination dataset: {dataset_id}")
print(f"   Tables in destination: {table_count}")
print(f"   Expected: {total_tables}")
print()

# Show first 10 tables as sample
print("📋 Sample tables in destination:")
for i, table in enumerate(dest_tables[:10]):
    print(f"   {i+1}. {table.table_id}")

if table_count > 10:
    print(f"   ... and {table_count - 10} more tables")

print()

# Verify we have the expected number of tables
if table_count >= total_tables - 5:  # Allow small variance
    print(f"✅ Checkpoint passed: Dataset copied successfully!")
    print(f"   Expected {total_tables} tables, found {table_count}")
else:
    print(f"⚠️  Warning: Expected {total_tables} tables but found {table_count}")
    print(f"   Some tables may not have been copied")

print()
print(f"🔗 View in Console:")
print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")

In [ ]:
# View dataset in BigQuery Console
print()
print("🔗 Quick Links:")
print(f"   Dataset: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}&p={PROJECT_ID}&page=dataset")
print(f"   Sample table: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")
print()
print("✅ Section 2 complete! Census Bureau ACS dataset is now available in your project.")